In [10]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

In [20]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

data_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=data_transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 28.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 288kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 7.15MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.04MB/s]


In [21]:
test_dataset = datasets.MNIST(root='./data',train=False,download=False,transform=data_transform)
test_loader = DataLoader(test_dataset,batch_size=256, shuffle=True)

In [17]:
class ProjectionHead(torch.nn.Module):
    def __init__(self, in_dim, out_dim):
        super(ProjectionHead, self).__init__()
        self.fc1 = torch.nn.Linear(in_dim, 512)
        self.fc2 = torch.nn.Linear(512, out_dim)

    def forward(self, x):
        x = torch.nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
    

In [29]:
class ContrastiveModel(torch.nn.Module):
    def __init__(self,encoder,projection_dim):
        super().__init__()
        self.encoder = encoder
        self.projection_head = ProjectionHead(encoder.fc.in_features, projection_dim)
        self.encoder.fc = torch.nn.Identity()
    def forward(self,x):
        encoded = self.encoder(x)
        return torch.nn.functional.normalize(self.projection_head(encoded),dim=1)


In [23]:
def contrastive_loss(z, temperature=0.5):
    N = z.shape[0]//2
    z = torch.nn.functional.normalize(z, dim=1)
    similarity_matrix = z @ z.T
    similarity_matrix.fill_diagonal_(-float('inf'))


    labels = torch.cat([torch.arange(N)+N,torch.arange(N)],dim=0)
    similarity_matrix = similarity_matrix/temperature
    return torch.nn.functional.cross_entropy(similarity_matrix,labels) 

In [34]:
transform = transforms.Compose([
    transforms.RandomResizedCrop(32),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.2),
])


In [35]:
encoder = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=False)
model = ContrastiveModel(encoder, projection_dim=128)

Using cache found in /Users/lukehoward/.cache/torch/hub/pytorch_vision_v0.10.0
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [36]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
for epoch in range(0,10):
    total_loss = 0
    model.train()

    for images, _ in train_loader:
        augmented_views = [transform(images) for _ in range(2)]
        projections1 = model(augmented_views[0])
        projections2 = model(augmented_views[1])

        projections = torch.cat([projections1, projections2], dim=0)
        loss = contrastive_loss(projections)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch [{epoch+1}/10], Loss: {total_loss/len(train_loader):.4f}')


In [ ]:
similarity1 = torch.cosine_similarity(x1,x2)
similarity2 = torch.cosine_similarity(x1,x3)
-np.log(np.exp(similarity1/0.5)/np.exp(similarity2/0.5))